# NBA Player Value Analyzer
## Notebook 02: Data Cleaning & Salary Integration

### Project Objective
The goal of this project is to identify NBA players who provide
the greatest on-court production relative to their salaries.

In this notebook, we will combine player performance statistics
from the 2024–25 NBA season with player salary information.

### Tasks
1. Import player statistics and salary datasets.
2. Examine data quality and formatting.
3. Clean player names and salary values.
4. Match salary information to player statistics.
5. Identify missing or unmatched players.
6. Save a combined dataset for future analysis.

### Data Sources
- NBA API: 2024–25 regular-season player statistics
- Public NBA salary dataset: 2024–25 player salaries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Identify the project directory
project_dir = Path.cwd()

# If running from notebooks/, move to the parent folder
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

print("Project directory:", project_dir)

# Define file paths
stats_path = project_dir / "data/processed/nba_players_cleaned.csv"
salary_path = project_dir / "data/raw/nba_salaries_2024_25.csv"

print("Stats file exists:", stats_path.exists())
print("Salary file exists:", salary_path.exists())

Project directory: /Users/charlieellis/nba-player-value-analyzer
Stats file exists: True
Salary file exists: True


In [3]:
# Load cleaned NBA statistics from Notebook 01
stats = pd.read_csv(stats_path)

# Load raw salary dataset
salaries = pd.read_csv(salary_path)

print("Player statistics rows:", len(stats))
print("Salary dataset rows:", len(salaries))

print("\nPLAYER STATISTICS:")
display(stats.head())

print("\nPLAYER SALARIES:")
display(salaries.head())

Player statistics rows: 569
Salary dataset rows: 563

PLAYER STATISTICS:


,player_id,player_name,team,age,games_played,minutes_per_game,points_per_game,rebounds_per_game,assists_per_game,steals_per_game,...,FG3_PCT,FT_PCT,FGM,FGA,FG3M,FG3A,FTM,FTA,TEAM_COUNT,TS_PCT
0,1630639,A.J. Lawson,TOR,24.0,26,18.7,9.1,3.3,1.2,0.5,...,0.327,0.683,3.1,7.3,1.3,3.9,1.7,2.4,1,0.541981
1,1631260,AJ Green,MIL,25.0,73,22.7,7.4,2.4,1.5,0.5,...,0.427,0.815,2.5,5.8,2.1,5.0,0.3,0.4,1,0.620584
2,1642358,AJ Johnson,WAS,20.0,29,22.0,7.6,2.0,2.6,0.4,...,0.267,0.865,2.8,7.3,0.8,3.1,1.1,1.3,2,0.479763
3,203932,Aaron Gordon,DEN,29.0,51,28.4,14.7,4.8,3.2,0.5,...,0.436,0.810,5.2,9.7,1.5,3.4,2.8,3.5,1,0.649576
4,1628988,Aaron Holiday,HOU,28.0,62,12.8,5.5,1.3,1.3,0.3,...,0.398,0.829,1.9,4.3,1.2,2.9,0.5,0.7,1,0.594322



PLAYER SALARIES:


,Player,Team,Salary
0,Stephen Curry,GSW,"$55,761,216"
1,Joel Embiid,PHI,"$51,415,938"
2,Nikola Jokic,DEN,"$51,415,938"
3,Kevin Durant,PHO,"$51,179,021"
4,Bradley Beal,PHO,"$50,203,930"


In [4]:
# Display salary column names
print("SALARY DATASET COLUMNS:")
print(salaries.columns.tolist())

# Inspect data types
print("\nDATA TYPES:")
print(salaries.dtypes)

# Check missing values
print("\nMISSING VALUES:")
print(salaries.isnull().sum())

# Check dataset dimensions
print("\nDATASET SHAPE:")
print(salaries.shape)

SALARY DATASET COLUMNS:
['Player', 'Team', 'Salary']

DATA TYPES:
Player    str
Team      str
Salary    str
dtype: object

MISSING VALUES:
Player    0
Team      0
Salary    0
dtype: int64

DATASET SHAPE:
(563, 3)


In [5]:
# Check exact duplicate rows
print("Exact duplicate rows:", salaries.duplicated().sum())

# Preview the first 10 records
display(salaries.head(10))

Exact duplicate rows: 27


,Player,Team,Salary
0,Stephen Curry,GSW,"$55,761,216"
1,Joel Embiid,PHI,"$51,415,938"
2,Nikola Jokic,DEN,"$51,415,938"
3,Kevin Durant,PHO,"$51,179,021"
4,Bradley Beal,PHO,"$50,203,930"
5,Jaylen Brown,BOS,"$49,205,800"
6,Devin Booker,PHO,"$49,205,800"
7,Karl-Anthony Towns,NYK,"$49,205,800"
8,Paul George,PHI,"$49,205,800"
9,Kawhi Leonard,LAC,"$49,205,800"


## Salary Data Cleaning

The salary dataset contains duplicate records and stores
salary values as formatted text.

We will remove exact duplicates, convert salaries into
numeric values, standardize player names, and investigate
remaining duplicate player records.

In [6]:
# Make a copy to preserve the original data
salaries_clean = salaries.copy()

# Remove exact duplicate rows
salaries_clean = salaries_clean.drop_duplicates()

print("Original rows:", len(salaries))
print("Rows after cleaning:", len(salaries_clean))
print("Duplicates removed:", len(salaries) - len(salaries_clean))

Original rows: 563
Rows after cleaning: 536
Duplicates removed: 27


In [7]:
# Remove dollar signs and commas
salaries_clean["Salary"] = (
    salaries_clean["Salary"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

# Convert to numeric values
salaries_clean["Salary"] = pd.to_numeric(
    salaries_clean["Salary"],
    errors="coerce"
)

# Rename columns
salaries_clean = salaries_clean.rename(columns={
    "Player": "player_name",
    "Team": "salary_team",
    "Salary": "salary"
})

print("Missing salary values:",
      salaries_clean["salary"].isna().sum())

display(salaries_clean.head())

Missing salary values: 0


,player_name,salary_team,salary
0,Stephen Curry,GSW,55761216
1,Joel Embiid,PHI,51415938
2,Nikola Jokic,DEN,51415938
3,Kevin Durant,PHO,51179021
4,Bradley Beal,PHO,50203930


In [8]:
# Find players with multiple salary records
duplicate_players = salaries_clean[
    salaries_clean.duplicated(
        subset=["player_name"],
        keep=False
    )
].sort_values("player_name")

print("Players with multiple salary records:",
      duplicate_players["player_name"].nunique())

display(duplicate_players)

Players with multiple salary records: 19


,player_name,salary_team,salary
368,Alex Len,WAS,2831348
462,Alex Len,LAL,2831348
26,Ben Simmons,BRK,40011909
460,Ben Simmons,LAC,40011909
515,Chuma Okeke,PHI,355687
534,Chuma Okeke,CLE,355687
537,Colin Castleton,TOR,421081
539,Colin Castleton,PHI,421081
312,David Roddy,ATL,2967212
527,David Roddy,PHI,2967212


## Player Name Standardization

Player names may be formatted differently across datasets.

To improve matching accuracy, we normalize names by:
- Converting text to lowercase
- Removing punctuation
- Removing accent marks
- Standardizing whitespace

We preserve the original player names for display.

In [10]:
import unicodedata
import re

def normalize_name(name):
    if pd.isna(name):
        return ""

    # Remove accents
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(
        char for char in name
        if not unicodedata.combining(char)
    )

    # Convert to lowercase
    name = name.lower()

    # Remove punctuation
    name = re.sub(r"[^a-z0-9\s]", "", name)

    # Remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name


# Apply to both datasets
stats["name_clean"] = stats["player_name"].apply(normalize_name)

salaries_clean["name_clean"] = (
    salaries_clean["player_name"].apply(normalize_name)
)

# Preview results
display(
    stats[["player_name", "name_clean"]].head(10)
)

,player_name,name_clean
0,A.J. Lawson,aj lawson
1,AJ Green,aj green
2,AJ Johnson,aj johnson
3,Aaron Gordon,aaron gordon
4,Aaron Holiday,aaron holiday
5,Aaron Nesmith,aaron nesmith
6,Aaron Wiggins,aaron wiggins
7,Adam Flagler,adam flagler
8,Adama Sanogo,adama sanogo
9,Adem Bona,adem bona


In [11]:
# Check name collisions in stats
stats_duplicates = stats[
    stats.duplicated("name_clean", keep=False)
]

# Check name collisions in salary data
salary_duplicates = salaries_clean[
    salaries_clean.duplicated("name_clean", keep=False)
]

print("Duplicate normalized names in stats:",
      stats_duplicates["name_clean"].nunique())

print("Duplicate normalized names in salaries:",
      salary_duplicates["name_clean"].nunique())

display(
    salary_duplicates[
        ["player_name", "salary_team", "salary", "name_clean"]
    ].sort_values("name_clean")
)

Duplicate normalized names in stats: 0
Duplicate normalized names in salaries: 19


,player_name,salary_team,salary,name_clean
368,Alex Len,WAS,2831348,alex len
462,Alex Len,LAL,2831348,alex len
26,Ben Simmons,BRK,40011909,ben simmons
460,Ben Simmons,LAC,40011909,ben simmons
515,Chuma Okeke,PHI,355687,chuma okeke
534,Chuma Okeke,CLE,355687,chuma okeke
537,Colin Castleton,TOR,421081,colin castleton
539,Colin Castleton,PHI,421081,colin castleton
312,David Roddy,ATL,2967212,david roddy
527,David Roddy,PHI,2967212,david roddy


In [12]:
# Check whether players have conflicting salary amounts
salary_conflicts = (
    salaries_clean
    .groupby("name_clean")["salary"]
    .nunique()
)

salary_conflicts = salary_conflicts[
    salary_conflicts > 1
]

print("Players with conflicting salaries:", len(salary_conflicts))

display(salary_conflicts)

Players with conflicting salaries: 0


Series([], Name: salary, dtype: int64)

In [13]:
# Combine salary records for players listed on multiple teams

salaries_final = (
    salaries_clean
    .groupby("name_clean", as_index=False)
    .agg(
        player_name=("player_name", "first"),
        salary=("salary", "first"),
        salary_teams=(
            "salary_team",
            lambda x: ", ".join(sorted(set(x)))
        )
    )
)

print("Original salary rows:", len(salaries))
print("After exact duplicate removal:", len(salaries_clean))
print("Final unique salary records:", len(salaries_final))

print(
    "Duplicate normalized names remaining:",
    salaries_final["name_clean"].duplicated().sum()
)

display(salaries_final.head())

Original salary rows: 563
After exact duplicate removal: 536
Final unique salary records: 517
Duplicate normalized names remaining: 0


,name_clean,player_name,salary,salary_teams
0,aaron gordon,Aaron Gordon,22841455,DEN
1,aaron holiday,Aaron Holiday,4668000,HOU
2,aaron nesmith,Aaron Nesmith,11000000,IND
3,aaron wiggins,Aaron Wiggins,10514017,OKC
4,adem bona,Adem Bona,1157153,PHI


In [14]:
display(
    salaries_final[
        salaries_final["name_clean"] == "russell westbrook"
    ]
)

,name_clean,player_name,salary,salary_teams
441,russell westbrook,Russell Westbrook,5631296,"DEN, UTA"


In [15]:
# Merge cleaned player statistics with salary information

merged = stats.merge(
    salaries_final[
        ["name_clean", "salary", "salary_teams"]
    ],
    on="name_clean",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Total player records:", len(merged))

print("\nMerge results:")
print(merged["_merge"].value_counts())

print("\nPlayers with salaries:")
print(merged["salary"].notna().sum())

display(
    merged[
        [
            "player_name",
            "team",
            "salary_teams",
            "points_per_game",
            "TS_PCT",
            "salary"
        ]
    ].head(10)
)

Total player records: 569

Merge results:
_merge
both          481
left_only      88
right_only      0
Name: count, dtype: int64

Players with salaries:
481


,player_name,team,salary_teams,points_per_game,TS_PCT,salary
0,A.J. Lawson,TOR,TOR,9.1,0.541981,100000.0
1,AJ Green,MIL,MIL,7.4,0.620584,2120693.0
2,AJ Johnson,WAS,WAS,7.6,0.479763,2795294.0
3,Aaron Gordon,DEN,DEN,14.7,0.649576,22841455.0
4,Aaron Holiday,HOU,HOU,5.5,0.594322,4668000.0
5,Aaron Nesmith,IND,IND,12.0,0.653066,11000000.0
6,Aaron Wiggins,OKC,OKC,12.0,0.595704,10514017.0
7,Adam Flagler,OKC,NaN,1.8,0.335467,NaN
8,Adama Sanogo,CHI,NaN,2.0,0.571429,NaN
9,Adem Bona,PHI,PHI,5.8,0.713983,1157153.0


In [16]:
unmatched = merged[
    merged["_merge"] == "left_only"
].copy()

print("Players without salary matches:", len(unmatched))

print(
    "Match rate:",
    round(
        merged["salary"].notna().mean() * 100, 2
    ),
    "%"
)

display(
    unmatched[
        ["player_id", "player_name", "team", "games_played"]
    ].sort_values("player_name")
)

Players without salary matches: 88
Match rate: 84.53 %


,player_id,player_name,team,games_played
7,1641745,Adam Flagler,OKC,37
8,1641766,Adama Sanogo,CHI,4
14,1642505,Alex Ducas,OKC,21
16,1642024,Alex Reese,PHI,15
18,1631214,Alondes Williams,LAC,1
...,...,...,...,...
546,1630623,Tyson Etienne,BKN,7
548,1642279,Ulrich Chomche,TOR,7
557,1630214,Xavier Tillman,BOS,33
558,1642530,Yuki Kawamura,MEM,22


In [17]:
# Standardize salary team abbreviations
team_mapping = {
    "BRK": "BKN",
    "CHO": "CHA",
    "PHO": "PHX"
}

def normalize_teams(team_string):
    if pd.isna(team_string):
        return []

    teams = team_string.split(", ")

    return [
        team_mapping.get(team, team)
        for team in teams
    ]

merged["salary_team_list"] = (
    merged["salary_teams"].apply(normalize_teams)
)

# Identify team mismatches
team_mismatches = merged[
    (merged["_merge"] == "both") &
    (
        ~merged.apply(
            lambda row: (
                row["team"] in row["salary_team_list"]
                or row["TEAM_COUNT"] > 1
            ),
            axis=1
        )
    )
]

print("Potential team mismatches:", len(team_mismatches))

display(
    team_mismatches[
        ["player_name", "team", "salary_teams", "salary"]
    ]
)

Potential team mismatches: 13


,player_name,team,salary_teams,salary
95,Cole Anthony,MEM,ORL,12900000.0
108,Daishen Nix,MIN,SAC,119972.0
116,Daniel Theis,NOP,OKC,2087519.0
140,Desmond Bane,ORL,MEM,34005250.0
162,E.J. Liddell,CHI,PHO,706898.0
231,Jalen McDaniels,SAC,"SAS, WAS",4861772.0
243,James Wiseman,IND,TOR,2237691.0
280,Johnny Davis,WAS,MEM,5291160.0
301,Josh Richardson,MIA,UTA,3051153.0
325,Kentavious Caldwell-Pope,MEM,ORL,22757000.0


## Investigating Unmatched Player Records

The initial merge matched 481 of 569 players (84.53%).

We will investigate unmatched records to distinguish between:
- Differences in player-name formatting
- Players missing from the salary source
- Players with short-term or two-way contracts

Potential fuzzy matches will be reviewed manually.

In [18]:
from difflib import get_close_matches

# Players without salary matches
unmatched = merged[
    merged["_merge"] == "left_only"
].copy()

# Names available in the salary dataset
salary_names = salaries_final["name_clean"].tolist()

# Find possible matches
unmatched["possible_match"] = (
    unmatched["name_clean"].apply(
        lambda name: get_close_matches(
            name,
            salary_names,
            n=1,
            cutoff=0.75
        )
    )
)

# Extract first suggested match
unmatched["possible_match"] = (
    unmatched["possible_match"].apply(
        lambda x: x[0] if x else None
    )
)

display(
    unmatched[
        [
            "player_name",
            "team",
            "games_played",
            "possible_match"
        ]
    ].sort_values("player_name")
)

,player_name,team,games_played,possible_match
7,Adam Flagler,OKC,37,NaN
8,Adama Sanogo,CHI,4,NaN
14,Alex Ducas,OKC,21,alex caruso
16,Alex Reese,PHI,15,NaN
18,Alondes Williams,LAC,1,jalen williams
...,...,...,...,...
546,Tyson Etienne,BKN,7,NaN
548,Ulrich Chomche,TOR,7,NaN
557,Xavier Tillman,BOS,33,xavier tillman sr
558,Yuki Kawamura,MEM,22,NaN


In [19]:
# Estimate total season minutes
unmatched["total_minutes_est"] = (
    unmatched["games_played"] *
    unmatched["minutes_per_game"]
)

# Sort unmatched players by minutes played
unmatched_priority = unmatched.sort_values(
    "total_minutes_est",
    ascending=False
)

display(
    unmatched_priority[
        [
            "player_name",
            "team",
            "games_played",
            "minutes_per_game",
            "total_minutes_est",
            "possible_match"
        ]
    ].head(25)
)

,player_name,team,games_played,minutes_per_game,total_minutes_est,possible_match
274,Jimmy Butler III,GSW,55,31.7,1743.5,jimmy butler
474,Ronald Holland II,DET,81,15.6,1263.6,ron holland
54,Brandon Boston,NOP,42,23.6,991.2,brandon boston jr
312,KJ Simpson,CHA,36,23.4,842.4,NaN
396,Micah Potter,UTA,38,18.6,706.8,NaN
521,Tosan Evbuomwan,BKN,28,23.7,663.6,NaN
265,Jeff Dowtin Jr.,PHI,41,15.1,619.1,NaN
329,Kessler Edwards,DAL,40,15.2,608.0,NaN
535,Tristan Vukcevic,WAS,35,14.6,511.0,NaN
237,Jamal Cain,NOP,37,13.6,503.2,NaN


In [20]:
# Calculate approximate total minutes for all players
merged["total_minutes_est"] = (
    merged["games_played"] *
    merged["minutes_per_game"]
)

# Players meeting approximate playing-time threshold
eligible = merged[
    merged["total_minutes_est"] >= 1000
].copy()

print("Players with 1000+ estimated minutes:", len(eligible))

print(
    "Eligible players with salaries:",
    eligible["salary"].notna().sum()
)

print(
    "Eligible players missing salaries:",
    eligible["salary"].isna().sum()
)

print(
    "Eligible salary coverage:",
    round(
        eligible["salary"].notna().mean() * 100,
        2
    ),
    "%"
)

Players with 1000+ estimated minutes: 270
Eligible players with salaries: 268
Eligible players missing salaries: 2
Eligible salary coverage: 99.26 %


## Manual Name Matching

After normalizing player names and merging the datasets,
two players meeting our 1,000-minute threshold remained
unmatched due to naming differences.

The following matches were manually verified:

- Jimmy Butler III → Jimmy Butler
- Ronald Holland II → Ron Holland

We use an explicit mapping to preserve matching accuracy
without introducing incorrect fuzzy matches.

In [22]:
# Verified player name mappings
name_corrections = {
    "jimmy butler iii": "jimmy butler",
    "ronald holland ii": "ron holland",
    "brandon boston jr": "brandon boston",
    "gg jackson ii": "gg jackson",
    "robert williams": "robert williams",
}

# Start with the original normalized name
stats["name_match"] = stats["name_clean"].copy()

# Apply manually verified corrections
stats["name_match"] = (
    stats["name_match"]
    .replace(name_corrections)
)

# Show corrected names
display(
    stats[
        stats["player_name"].isin([
            "Jimmy Butler III",
            "Ronald Holland II",
            "Brandon Boston",
            "GG Jackson",
            "Robert Williams III"
        ])
    ][
        ["player_name", "name_clean", "name_match"]
    ]
)

,player_name,name_clean,name_match
54,Brandon Boston,brandon boston,brandon boston
172,GG Jackson,gg jackson,gg jackson
274,Jimmy Butler III,jimmy butler iii,jimmy butler
472,Robert Williams III,robert williams iii,robert williams iii
474,Ronald Holland II,ronald holland ii,ron holland


In [23]:
salaries_final["name_match"] = salaries_final["name_clean"]

merged_final = stats.merge(
    salaries_final[["name_match", "salary", "salary_teams"]],
    on="name_match",
    how="left",
    validate="one_to_one",
    indicator=True
)

print(merged_final["_merge"].value_counts())

_merge
both          483
left_only      86
right_only      0
Name: count, dtype: int64


In [24]:
merged_final["total_minutes_est"] = (
    merged_final["games_played"] *
    merged_final["minutes_per_game"]
)

eligible_players = merged_final[
    merged_final["total_minutes_est"] >= 1000
]

print("Eligible players:", len(eligible_players))
print("With salaries:", eligible_players["salary"].notna().sum())
print("Missing salaries:", eligible_players["salary"].isna().sum())

Eligible players: 270
With salaries: 270
Missing salaries: 0


## Final Dataset Export

After cleaning the salary dataset and resolving player-name
differences, all 270 players meeting our estimated 1,000-minute
eligibility threshold have matching salary records.

We will preserve all 569 player records in the combined dataset,
including those with missing salary information.

This dataset will be used for exploratory analysis and
development of our player value model.

In [25]:
# Create a copy of our merged dataset
final_df = merged_final.copy()

# Remove temporary columns used during merging
final_df = final_df.drop(
    columns=["_merge", "name_clean", "name_match"]
)

# Rename salary column for clarity
final_df = final_df.rename(
    columns={"salary": "salary_2024_25"}
)

# Preview final dataset
print("Final dataset shape:", final_df.shape)

display(final_df.head())

Final dataset shape: (569, 26)


,player_id,player_name,team,age,games_played,minutes_per_game,points_per_game,rebounds_per_game,assists_per_game,steals_per_game,...,FGA,FG3M,FG3A,FTM,FTA,TEAM_COUNT,TS_PCT,salary_2024_25,salary_teams,total_minutes_est
0,1630639,A.J. Lawson,TOR,24.0,26,18.7,9.1,3.3,1.2,0.5,...,7.3,1.3,3.9,1.7,2.4,1,0.541981,100000.0,TOR,486.2
1,1631260,AJ Green,MIL,25.0,73,22.7,7.4,2.4,1.5,0.5,...,5.8,2.1,5.0,0.3,0.4,1,0.620584,2120693.0,MIL,1657.1
2,1642358,AJ Johnson,WAS,20.0,29,22.0,7.6,2.0,2.6,0.4,...,7.3,0.8,3.1,1.1,1.3,2,0.479763,2795294.0,WAS,638.0
3,203932,Aaron Gordon,DEN,29.0,51,28.4,14.7,4.8,3.2,0.5,...,9.7,1.5,3.4,2.8,3.5,1,0.649576,22841455.0,DEN,1448.4
4,1628988,Aaron Holiday,HOU,28.0,62,12.8,5.5,1.3,1.3,0.3,...,4.3,1.2,2.9,0.5,0.7,1,0.594322,4668000.0,HOU,793.6


In [26]:
from pathlib import Path

# Identify project directory
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Create processed data folder if needed
output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Save final dataset
output_path = output_dir / "nba_player_values_base.csv"

final_df.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("File location:", output_path)
print("Total players:", len(final_df))

Dataset saved successfully!
File location: /Users/charlieellis/nba-player-value-analyzer/data/processed/nba_player_values_base.csv
Total players: 569


In [27]:
# Reload the saved dataset
check_df = pd.read_csv(output_path)

print("Saved rows:", len(check_df))
print("Saved columns:", len(check_df.columns))

print("Players with salaries:",
      check_df["salary_2024_25"].notna().sum())

# Confirm the player IDs remain unique
assert len(check_df) == 569
assert check_df["player_id"].is_unique

print("Validation passed!")

Saved rows: 569
Saved columns: 26
Players with salaries: 483
Validation passed!
